# 🌲🌊 숲으로 바다로 — 팀원 온보딩 노트북

> 도시를 떠나 자연을 만끽하고 싶은 여러분에게, 떠나고 싶은 곳의 공공 편의시설·안전정보를 알려주는 서비스.

## 사용법 — 이 노트북 하나면 끝!
이 `.ipynb` 파일만 받아서 **위에서부터 셀을 하나씩 실행**(`Shift+Enter`)하면
**클론 → 의존성 설치 → 환경 점검**까지 자동으로 됩니다.
(주피터에서 위 메뉴 `Run` → `Run All Cells` 로 한 번에 돌려도 됩니다.)

### 먼저 딱 두 가지만 준비
1. **Git 설치** — 없으면 https://git-scm.com 에서 설치
2. **GitHub 초대 수락** — 비공개 레포라, 리더(D)가 여러분을 Collaborator로 초대합니다.
   초대 메일(또는 github.com/notifications)에서 **수락**하세요. 안 하면 클론이 안 됩니다.

> 파이썬 3.10+ 와 주피터(또는 VS Code)만 있으면 됩니다.

## 0. 프로젝트 받기 (클론) — 자동

아래 셀을 실행하면 프로젝트를 내려받습니다. **비공개 레포라 GitHub 로그인 창이 뜰 수 있어요** — 로그인하면 됩니다.
(이미 클론된 폴더 안에서 이 노트북을 열었다면 자동으로 건너뜁니다.)

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/shind0716-cpu/camping-map.git'
REPO_DIR = 'camping-map'

def is_repo_root(p):
    return (Path(p) / 'backend' / 'main.py').exists() and (Path(p) / 'data' / 'facilities.db').exists()

if is_repo_root(Path.cwd()):
    print('✅ 이미 프로젝트 폴더 안에서 실행 중입니다. (클론 생략)')
elif Path(REPO_DIR).exists():
    os.chdir(REPO_DIR)
    print(f'✅ 기존 클론 폴더로 이동했습니다 → {Path.cwd()}')
else:
    print('⏳ 프로젝트를 클론합니다... (로그인 창이 뜨면 GitHub 로그인)')
    r = subprocess.run(['git', 'clone', REPO_URL], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if Path(REPO_DIR).exists():
        os.chdir(REPO_DIR)
        print(f'✅ 클론 완료 → {Path.cwd()}')
    else:
        print('❌ 클론 실패. 확인하세요:')
        print('   1) 리더(D)에게 GitHub Collaborator 초대를 받고 수락했는지')
        print('   2) Git 설치 + GitHub 로그인(자격증명) 상태')
        print('   3) (수동) 터미널에서:  git clone ' + REPO_URL)

In [ ]:
# 파이썬 의존성 설치 (fastapi·uvicorn·pyproj 등). 1~2분 걸릴 수 있어요.
import sys, subprocess
print('의존성 설치 중...')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
                   capture_output=True, text=True)
print('✅ 설치 완료' if r.returncode == 0 else '⚠️ 설치 중 문제:\n' + r.stderr[-800:])

## 1. 환경 점검 — 파일·키가 다 있나

In [ ]:
ROOT = Path.cwd()
checks = {
    'backend/main.py (백엔드)': (ROOT / 'backend' / 'main.py').exists(),
    'frontend/index.html (프론트)': (ROOT / 'frontend' / 'index.html').exists(),
    'data/facilities.db (DB)': (ROOT / 'data' / 'facilities.db').exists(),
    '.env (API 키)': (ROOT / '.env').exists(),
    'requirements.txt': (ROOT / 'requirements.txt').exists(),
}
for name, ok in checks.items():
    print(('✅' if ok else '❌ 없음!'), name)

if not checks['.env (API 키)']:
    print('\n→ .env 가 없으면 지도·실시간 기능이 안 됩니다. (비공개 레포라 clone에 포함돼 있어야 정상)')

In [ ]:
# .env 의 키가 채워져 있는지 확인 (값은 보안상 가리고 존재 여부만)
def load_env(path='.env'):
    env = {}
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            env[k.strip()] = v.split('#')[0].strip()
    return env

env = load_env()
need = ['DATA_GO_KR_KEY', 'KAKAO_REST_KEY', 'KAKAO_JS_KEY', 'JEJU_ITS_CODE']
for k in need:
    v = env.get(k, '')
    filled = bool(v) and '키' not in v and '여기' not in v
    print(('✅' if filled else '❌ 비어있음'), k, f'(...{v[-4:]})' if filled else '')

gpt = env.get('GPT_MINI_API_KEY', '')
print(('✅' if gpt and '여기' not in gpt else '⏳ 아직 없음 (수요일 도착 예정)'), 'GPT_MINI_API_KEY')

## 2. 데이터 살펴보기 — 무슨 데이터가 몇 건 있나

In [ ]:
import sqlite3

LABEL = {
    'restroom': '🚻 화장실', 'parking_public': '🚗 공영주차장', 'ev_charger': '🔌 전기차충전소',
    'camping': '⛺ 야영장', 'festival': '🎉 문화축제', 'tourinfo': 'ℹ️ 관광안내소',
    'cleanhouse': '♻️ 클린하우스', 'parking': '🅿️ 둔치주차장', 'beach': '🏖️ 해수욕장',
    'cctv': '📹 안전CCTV', 'pharmacy': '💊 심야약국',
}
con = sqlite3.connect('data/facilities.db')
con.row_factory = sqlite3.Row
rows = con.execute('SELECT category, COUNT(*) n FROM facilities GROUP BY category ORDER BY n DESC').fetchall()
total = 0
for r in rows:
    total += r['n']
    print(f"  {LABEL.get(r['category'], r['category']):16} {r['n']:>7,}")
print(f"  {'합계':16} {total:>7,}")

In [ ]:
# 특정 지역 조회 예시 — '서귀포 공영주차장' (추천 로직이 쓰는 방식)
import json

q = '''SELECT name, lat, lon, road_addr, extra FROM facilities
       WHERE category='parking_public' AND road_addr LIKE '%서귀포%' LIMIT 5'''
for r in con.execute(q):
    e = json.loads(r['extra'])
    print(f"  {r['name']}  [{e.get('gubun')}/{e.get('type')}, {e.get('capacity')}면]  @ {r['lat']:.4f},{r['lon']:.4f}")

In [ ]:
# 지도 API가 쓰는 bbox(화면 영역) 조회 — 백엔드 /api/facilities 와 같은 쿼리
def in_bbox(category, sw_lat, sw_lng, ne_lat, ne_lng, limit=5):
    sql = '''SELECT name, lat, lon FROM facilities
             WHERE category=? AND lat BETWEEN ? AND ? AND lon BETWEEN ? AND ? LIMIT ?'''
    return con.execute(sql, (category, sw_lat, ne_lat, sw_lng, ne_lng, limit)).fetchall()

# 서귀포 시내 대략 범위의 화장실
for r in in_bbox('restroom', 33.24, 126.55, 33.27, 126.58):
    print(' ', r['name'], '@', round(r['lat'], 4), round(r['lon'], 4))

## 3. 서버 실행해보기

아래는 **터미널**에서 실행하세요 (프로젝트 폴더 `camping-map` 안에서):

```bash
uvicorn backend.main:app --reload --port 8000
```

→ 브라우저에서 **http://localhost:8000** 접속.
- 검색창(숲/바다 배경) 화면이 뜨면 프론트 OK
- '🗺️ 직접 지도에서 찾아보기' → 마커가 뜨면 DB·카카오키 OK

> ⚠️ 반드시 **8000 포트**. 카카오맵이 `localhost:8000` 도메인에만 등록돼 있습니다.

아래 셀은 서버가 떠 있을 때 `/api/health` 를 찔러 정상 여부를 확인합니다 (서버 먼저 실행 후 실행).

In [ ]:
import urllib.request, json
try:
    with urllib.request.urlopen('http://localhost:8000/api/health', timeout=5) as r:
        d = json.load(r)
    print('✅ 서버 정상:', d.get('status'), '| DB:', d.get('db'), '| 좌표 있는 시설:', d.get('with_coord'))
except Exception as e:
    print('⏳ 서버가 아직 안 떠 있습니다. 터미널에서 uvicorn 실행 후 이 셀을 다시 실행하세요.')
    print('   (', e, ')')

## 4. 내 담당은 어디를 만지면 되나 (역할별 폴더)

| 팀원 | 역할 | 주 작업 폴더 | 참고 문서 |
|---|---|---|---|
| **A** | 프론트 & 통합 | `frontend/` (index.html·app.js·style.css) | — |
| **B** | AI / RAG | `rag/` (신규) — 지식베이스는 `data/raw/*.jsonl` | `docs/recommendation-pipeline.md` |
| **C** | 추천 백엔드 + 배포 | `backend/main.py`, `Dockerfile` | `docs/recommendation-pipeline.md` |
| **D** | 데이터 · 일정 · 발표 | `data_pipeline/`, `docs/` | `docs/data-management.md` |

**꼭 읽을 문서**
- `docs/CONTRIBUTING.md` — 브랜치·커밋·협업 규칙 (⚠️ `.env`가 이제 레포에 있으니 **레포를 절대 public 으로 바꾸지 말 것**. 새 키를 실수로 커밋만 안 하면 됨)
- `docs/TODO.md` — 7/7~7/9 일자별 할 일 (본인 파트 확인)
- `docs/recommendation-pipeline.md` — 추천 점수화 + 질문 학습 루프 설계 (B·C 필독)

### RAG 담당(B)용 — 지식베이스 원문 위치
`data/raw/` 안: `camping.jsonl`(야영장 소개) · `festival.jsonl`·`festival_std.jsonl`(축제) · `city_tour.jsonl` · `theme_park.jsonl`(아이 놀거리) · `hanok_stay.jsonl`(촌캉스) · `tourinfo.jsonl`

In [ ]:
# RAG 원문 미리보기 (B 담당) — 축제 데이터 1건 구조 확인
with open('data/raw/festival_std.jsonl', encoding='utf-8') as f:
    sample = json.loads(f.readline())
print(json.dumps(sample, ensure_ascii=False, indent=2)[:600])

## ✅ 온보딩 끝!

여기까지 ✅ 가 다 떴으면 개발 시작 준비 완료입니다.

막히면: 30분 이상 혼자 붙잡지 말고 **카톡방 + 화면공유**. 진행 상황은 `docs/TODO.md` 체크박스로 표시하고 커밋하세요.

> 브랜치 규칙: `main` 직접 push 금지 → `feat/내작업` 브랜치 → PR. push 전 `git pull` 먼저!